## 1. Environment Setup <a name='setup'></a>

## 1. Environment Setup <a name='setup'></a>

In [1]:
import os

BASE_DIR  = '/content/press_start'
DATA_DIR  = os.path.join(BASE_DIR, 'data')
CLEAN_DIR = os.path.join(BASE_DIR, 'clean')
PLOT_DIR  = os.path.join(BASE_DIR, 'plots')

for d in [DATA_DIR, CLEAN_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Directories ready ✓")

Directories ready ✓


In [2]:
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn           as sns

from sklearn.linear_model    import LinearRegression
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import mean_squared_error, r2_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

RANDOM_STATE = 42
TEST_SIZE    = 0.2
TARGET_COL   = 'Estimated_Owners'   # sales proxy — midpoint of owner range
COLORS       = ['#4C72B0', '#DD8452', '#55A868']

print("Libraries loaded ✓")

Libraries loaded ✓


## 2. Problem Statement <a name='problem'></a>

The video game industry has seen consistent price increases, with AAA titles now regularly launching at USD $70. Yet publishers and developers lack a data-driven framework for determining whether these prices maximise sales or actively suppress them.

**Central Question:** Given a game's features — including its price — can we predict how many copies it will sell, and use that model to find the price point that maximises sales?

**Sub-questions:**
- How strongly does price correlate with sales volume?
- Do holiday/seasonal releases sell significantly more copies?
- What is the optimal price point predicted by each model?
- Are games currently priced above or below that optimum — i.e., are they overpriced?

**Problem Type:** Regression — predicting a continuous numerical value (estimated owner count).

### Stakeholders

| Stakeholder | Need |
|-------------|------|
| Game publishers | Set a price that maximises revenue without suppressing sales |
| Independent developers | Understand how pricing decisions affect their market reach |
| Consumers | Understand whether current pricing reflects market demand |

### Why it Matters
A data-driven pricing model could save publishers from leaving money on the table through under-pricing, or losing customers through over-pricing. For indie developers, this insight is particularly valuable as pricing is one of the few levers fully within their control.


## 3. Dataset Download <a name='download'></a>

### Datasets Used

All three datasets are sourced from Kaggle and focus on the Steam platform, which provides the richest publicly available combination of **price**, **sales proxy (estimated owners)**, **genre**, **review scores**, and **release dates**.

| # | Kaggle Slug | Description | Key Columns |
|---|-------------|-------------|-------------|
| 1 | `nikdavis/steam-store-games` | ~27,000 Steam games scraped 2019. Clean, well-structured. | `price`, `owners` (range), `release_date`, `genres`, `positive_ratings`, `negative_ratings` |
| 2 | `fronkongames/steam-games-dataset` | 110,000+ Steam games, most comprehensive available. | `price`, `estimated_owners` (range), `release_date`, `genres`, `positive`, `negative`, `peak_ccu` |
| 3 | `trolukovich/steam-games-complete-dataset` | ~40,000 games with detailed feature coverage and price history. | `original_price`, `discount_price`, `genre`, `release_date`, `developer`, `publisher` |

**Why Steam?**  
Unlike the classic vgsales datasets, Steam data includes **price** as a first-class feature — which is essential for the price optimisation analysis. The owner range (e.g. "20000 - 50000") serves as our sales proxy; we take the midpoint as the target value. Steam also provides release dates at the month level, enabling seasonality analysis.

---
Get your Kaggle API token at: **kaggle.com → Account → API → Create New Token**


In [3]:
import os

# ── Fill in your own Kaggle credentials ──────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = 'matthewjordansingh'   # e.g. 'matthew_singh'
os.environ['KAGGLE_KEY']      = 'KGAT_1ab7ba8220d09570317109db252a4300'        # the key string from your token
# ─────────────────────────────────────────────────────────────────────────────

!pip install kaggle -q
print("Kaggle configured ✓")

Kaggle configured ✓


In [4]:
datasets_to_download = {
    'dataset1': 'nikdavis/steam-store-games',
    'dataset2': 'fronkongames/steam-games-dataset',
    'dataset3': 'trolukovich/steam-games-complete-dataset',
}

for key, slug in datasets_to_download.items():
    out = os.path.join(DATA_DIR, key)
    os.makedirs(out, exist_ok=True)
    print(f"Downloading {slug} ...")
    !kaggle datasets download -d {slug} -p {out} --unzip -q
    print(f"  → {os.listdir(out)}\n")

print("All downloads complete ✓")

Dataset URL: https://www.kaggle.com/datasets/nikdavis/steam-store-games
License(s): Attribution 4.0 International (CC BY 4.0)
  → ['steam_support_info.csv', 'steam_requirements_data.csv', 'steam_media_data.csv', 'steamspy_tag_data.csv', 'steam_description_data.csv', 'steam.csv']

Dataset URL: https://www.kaggle.com/datasets/fronkongames/steam-games-dataset
License(s): MIT
  → ['games.json', 'games.csv']

Dataset URL: https://www.kaggle.com/datasets/trolukovich/steam-games-complete-dataset
License(s): CC0-1.0
  → ['steam_games.csv']

All downloads complete ✓


## 4. Data Inspection <a name='inspect'></a>

We inspect the raw files before any cleaning to understand column names, data types, and missing value patterns. **Update the paths below** after running the download cell — check what CSV files were created.


In [5]:
# ── Update filenames if needed based on download output above ─────────────────
RAW_PATH_1 = os.path.join(DATA_DIR, 'dataset1', 'steam.csv')
RAW_PATH_2 = os.path.join(DATA_DIR, 'dataset2', 'games.csv')
RAW_PATH_3 = os.path.join(DATA_DIR, 'dataset3', 'steam_games.csv')

raw_paths = {
    'Dataset 1 (nikdavis)':     RAW_PATH_1,
    'Dataset 2 (fronkongames)': RAW_PATH_2,
    'Dataset 3 (trolukovich)':  RAW_PATH_3,
}

raw_dfs = {}
for name, path in raw_paths.items():
    try:
        df = pd.read_csv(path)
        raw_dfs[name] = df
        print(f"\n{'='*60}")
        print(f"{name}  |  Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        display(df.head(3))
        print("\nMissing values (columns with nulls only):")
        m = df.isnull().sum()
        print(m[m > 0].to_string() if m.any() else "  None")
    except FileNotFoundError:
        print(f"\n[!] Not found: {path}  — check filename above")


Dataset 1 (nikdavis)  |  Shape: (27075, 18)
Columns: ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99



Missing values (columns with nulls only):
developer     1
publisher    14

Dataset 2 (fronkongames)  |  Shape: (122611, 39)
Columns: ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN



Missing values (columns with nulls only):
AppID                  1
About the game      8449
Reviews           110541
Header image          81
Website            72935
Support url        68469
Support email      22263
Metacritic url    118355
Score rank        122571
Notes             100153
Developers          8437
Publishers          8909
Categories          8953
Genres              8413
Tags               39265
Screenshots         6018
Movies            122611

Dataset 3 (trolukovich)  |  Shape: (40833, 20)
Columns: ['url', 'types', 'name', 'desc_snippet', 'recent_reviews', 'all_reviews', 'release_date', 'developer', 'publisher', 'popular_tags', 'game_details', 'languages', 'achievements', 'genre', 'game_description', 'mature_content', 'minimum_requirements', 'recommended_requirements', 'original_price', 'discount_price']


,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,popular_tags,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks","FPS,Gore,Action,Demons,Shooter,First-Person,Gr...","Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99
1,https://store.steampowered.com/app/578080/PLAY...,app,PLAYERUNKNOWN'S BATTLEGROUNDS,PLAYERUNKNOWN'S BATTLEGROUNDS is a battle roya...,"Mixed,(6,214),- 49% of the 6,214 user reviews ...","Mixed,(836,608),- 49% of the 836,608 user revi...","Dec 21, 2017",PUBG Corporation,"PUBG Corporation,PUBG Corporation","Survival,Shooter,Multiplayer,Battle Royale,PvP...","Multi-player,Online Multi-Player,Stats","English,Korean,Simplified Chinese,French,Germa...",37.0,"Action,Adventure,Massively Multiplayer",About This Game PLAYERUNKNOWN'S BATTLEGROUND...,Mature Content Description The developers de...,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",$29.99,NaN
2,https://store.steampowered.com/app/637090/BATT...,app,BATTLETECH,Take command of your own mercenary outfit of '...,"Mixed,(166),- 54% of the 166 user reviews in t...","Mostly Positive,(7,030),- 71% of the 7,030 use...","Apr 24, 2018",Harebrained Schemes,"Paradox Interactive,Paradox Interactive","Mechs,Strategy,Turn-Based,Turn-Based Tactics,S...","Single-player,Multi-player,Online Multi-Player...","English,French,German,Russian",128.0,"Action,Adventure,Strategy",About This Game From original BATTLETECH/Mec...,NaN,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",$39.99,NaN



Missing values (columns with nulls only):
types                           2
name                           16
desc_snippet                13221
recent_reviews              38127
all_reviews                 12363
release_date                 3179
developer                     343
publisher                    5100
popular_tags                 2945
game_details                  520
languages                      36
achievements                28639
genre                         438
game_description             2913
mature_content              37936
minimum_requirements        19764
recommended_requirements    19758
original_price               5311
discount_price              26290


## 5. Preprocessing <a name='preprocessing'></a>

Each dataset has different column names and structures. We standardise them around these core features:

| Column | Source | Notes |
|--------|--------|-------|
| `Price` | Direct from dataset | USD, 0 = free-to-play |
| `Estimated_Owners` | Parsed from range string | Midpoint of range e.g. "20000 - 50000" → 35000 |
| `Genre` | Label-encoded | Categorical |
| `Release_Month` | Parsed from release date | 1–12 |
| `Is_Holiday` | Engineered | 1 if release month is Oct, Nov, or Dec |
| `Positive_Ratio` | Computed | positive / (positive + negative) ratings |
| `DLC_Count` | Direct | Number of DLCs |

### Why these features?
- **Price** is the central feature for the optimisation analysis
- **Is_Holiday** addresses the TA's seasonality comment directly
- **Positive_Ratio** captures game quality — a confound we must control for
- **DLC_Count** reflects franchise depth and post-launch investment


In [6]:
def parse_owners(owners_str):
    """
    Convert owner range string to numeric midpoint.
    e.g. '20000 - 50000' → 35000
         '0 - 20000'     → 10000
    Returns NaN if unparseable.
    """
    try:
        owners_str = str(owners_str).replace(',', '').strip()
        if '-' in owners_str:
            parts = owners_str.split('-')
            lo, hi = float(parts[0].strip()), float(parts[1].strip())
            return (lo + hi) / 2
        return float(owners_str)
    except:
        return np.nan

def parse_release_month(date_str):
    """Extract month (1-12) from various date string formats."""
    try:
        return pd.to_datetime(str(date_str), infer_datetime_format=True).month
    except:
        return np.nan

def encode_col(df, col):
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[col] = le.fit_transform(df[col])
    return df

def clip_and_log(df, col, upper_q=0.99):
    cap = df[col].quantile(upper_q)
    n   = (df[col] > cap).sum()
    df[col + '_raw'] = df[col]
    df[col]          = df[col].clip(upper=cap)
    df[col]          = np.log1p(df[col])
    print(f"  Clipped {n} outlier(s) at {cap:,.0f} owners, applied log1p")
    return df

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET 1 — nikdavis/steam-store-games
# Columns of interest: price, owners, release_date, genres,
#                      positive_ratings, negative_ratings, developer, publisher
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset1(path):
    print("\nCleaning Dataset 1 (nikdavis)...")
    df = pd.read_csv(path)

    # ── Standardise column names (lowercase → Title) ──
    df.columns = [c.strip() for c in df.columns]
    rename = {
        'price':            'Price',
        'owners':           'Estimated_Owners',
        'genres':           'Genre',
        'release_date':     'Release_Date',
        'positive_ratings': 'Positive',
        'negative_ratings': 'Negative',
        'developer':        'Developer',
        'publisher':        'Publisher',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    df['Estimated_Owners'] = df['Estimated_Owners'].apply(parse_owners)
    df = df.dropna(subset=['Estimated_Owners', 'Price'])
    df = df[df['Price'] >= 0]

    before = len(df)
    df = df[df['Estimated_Owners'] > 0]
    print(f"  Dropped {before - len(df)} rows with 0 owners")

    df['Release_Month'] = df['Release_Date'].apply(parse_release_month)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)

    total = df['Positive'].fillna(0) + df['Negative'].fillna(0)
    df['Positive_Ratio'] = np.where(total > 0, df['Positive'].fillna(0) / total, 0.5)
    df['DLC_Count'] = 0   # not in this dataset

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [8]:
def clean_dataset2(path):
    print("\nCleaning Dataset 2 (fronkongames)...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename = {
        'Price':             'Price',
        'Estimated owners':  'Estimated_Owners',
        'Genres':            'Genre',
        'Release date':      'Release_Date',
        'Positive':          'Positive',
        'Negative':          'Negative',
        'DLC count':         'DLC_Count',
        'Peak CCU':          'Peak_CCU',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    df['Estimated_Owners'] = df['Estimated_Owners'].apply(parse_owners)
    df['Price']            = pd.to_numeric(df['Price'], errors='coerce')
    df = df.dropna(subset=['Estimated_Owners', 'Price'])
    df = df[df['Price'] >= 0]

    before = len(df)
    df = df[df['Estimated_Owners'] > 0]
    print(f"  Dropped {before - len(df)} rows with 0 owners")

    # Parse release date properly then fill missing months with 6 (June)
    df['Release_Date']  = pd.to_datetime(df['Release_Date'], errors='coerce')
    df['Release_Month'] = df['Release_Date'].dt.month.fillna(6).astype(int)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)

    total = df['Positive'].fillna(0) + df['Negative'].fillna(0)
    df['Positive_Ratio'] = np.where(total > 0, df['Positive'].fillna(0) / total, 0.5)

    if 'DLC_Count' not in df.columns:
        df['DLC_Count'] = 0

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET 3 — trolukovich/steam-games-complete-dataset
# Columns of interest: original_price, discount_price, genre,
#                      release_date, developer, publisher
# Target proxy: we engineer a popularity score from available signals
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset3(path):
    print("\nCleaning Dataset 3 (trolukovich)...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename = {
        'original_price':  'Price',
        'discount_price':  'Discount_Price',
        'genre':           'Genre',
        'release_date':    'Release_Date',
        'developer':       'Developer',
        'publisher':       'Publisher',
        'reviews_from_purchased_people': 'Review_Count',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    # Clean price — strip currency symbols, convert to float
    for col in ['Price', 'Discount_Price']:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                               .str.replace(r'[^0-9.]', '', regex=True))
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Price'])
    df = df[df['Price'] >= 0]

    # This dataset has no direct owner count.
    # We use discount depth as a proxy for price pressure:
    # discount_depth = (original - discount) / original
    # and retain Price as the main feature for the optimisation.
    # We create a synthetic Estimated_Owners from review counts if available.
    if 'Review_Count' in df.columns:
        df['Review_Count'] = pd.to_numeric(
            df['Review_Count'].astype(str).str.replace(r'[^0-9]', '', regex=True),
            errors='coerce').fillna(0)
        # Rough industry estimate: ~50 reviews per 1000 owners
        df['Estimated_Owners'] = df['Review_Count'] * 20
    else:
        # Fallback: estimate from price tier (rough proxy only)
        # Not ideal but keeps the dataset usable for the price analysis
        df['Estimated_Owners'] = np.where(df['Price'] == 0, 50000,
                                 np.where(df['Price'] < 10, 20000,
                                 np.where(df['Price'] < 30, 10000, 5000)))
        print("  [Note] No review/owner data — using price-tier proxy for owners.")
        print("  Dataset 3 results should be interpreted with this caveat.")

    df = df[df['Estimated_Owners'] > 0]
    df['Release_Month'] = df['Release_Date'].apply(parse_release_month)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)
    df['Positive_Ratio'] = 0.5   # not available
    df['DLC_Count']      = 0

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [10]:
df1 = clean_dataset1(RAW_PATH_1)
df2 = clean_dataset2(RAW_PATH_2)
df3 = clean_dataset3(RAW_PATH_3)

datasets = {
    'Dataset 1 (nikdavis)':     df1,
    'Dataset 2 (fronkongames)': df2,
    'Dataset 3 (trolukovich)':  df3,
}

# Save cleaned CSVs to Drive
df1.to_csv(os.path.join(CLEAN_DIR, 'dataset1_clean.csv'), index=False)
df2.to_csv(os.path.join(CLEAN_DIR, 'dataset2_clean.csv'), index=False)
df3.to_csv(os.path.join(CLEAN_DIR, 'dataset3_clean.csv'), index=False)
print("\nCleaned datasets saved ✓")


Cleaning Dataset 1 (nikdavis)...
  Dropped 0 rows with 0 owners
  Clipped 266 outlier(s) at 1,500,000 owners, applied log1p
  Final shape: (27075, 23)

Cleaning Dataset 2 (fronkongames)...
  Dropped 102935 rows with 0 owners
  Clipped 197 outlier(s) at 3,710 owners, applied log1p
  Final shape: (19676, 44)

Cleaning Dataset 3 (trolukovich)...
  [Note] No review/owner data — using price-tier proxy for owners.
  Dataset 3 results should be interpreted with this caveat.
  Clipped 0 outlier(s) at 20,000 owners, applied log1p
  Final shape: (32520, 26)

Cleaned datasets saved ✓


In [11]:
# Sanity check — confirm all key columns are present
FEATURE_COLS = ['Price', 'Genre', 'Is_Holiday', 'Positive_Ratio',
                'DLC_Count', 'Release_Month']

for name, df in datasets.items():
    present = [c for c in FEATURE_COLS if c in df.columns]
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    print(f"\n{name}  |  shape: {df.shape}")
    print(f"  Features present : {present}")
    if missing:
        print(f"  Features MISSING : {missing}")
    display(df[present + [TARGET_COL]].head(3))


Dataset 1 (nikdavis)  |  shape: (27075, 23)
  Features present : ['Price', 'Genre', 'Is_Holiday', 'Positive_Ratio', 'DLC_Count', 'Release_Month']


,Price,Genre,Is_Holiday,Positive_Ratio,DLC_Count,Release_Month,Estimated_Owners
0,7.19,2,1,0.973888,0,11,14.220976
1,3.99,2,0,0.839787,0,4,14.220976
2,3.99,2,0,0.895648,0,5,14.220976



Dataset 2 (fronkongames)  |  shape: (19676, 44)
  Features present : ['Price', 'Genre', 'Is_Holiday', 'Positive_Ratio', 'DLC_Count', 'Release_Month']


,Price,Genre,Is_Holiday,Positive_Ratio,DLC_Count,Release_Month,Estimated_Owners
3292190,0,894,0,0.50,0,6,0.693147
1934300,10,1251,0,0.90,0,6,2.197225
1540330,0,328,0,0.76,0,6,0.693147



Dataset 3 (trolukovich)  |  shape: (32520, 26)
  Features present : ['Price', 'Genre', 'Is_Holiday', 'Positive_Ratio', 'DLC_Count', 'Release_Month']


,Price,Genre,Is_Holiday,Positive_Ratio,DLC_Count,Release_Month,Estimated_Owners
0,19.99,3,0,0.5,0,5.0,9.210440
1,29.99,180,1,0.5,0,12.0,9.210440
2,39.99,213,0,0.5,0,4.0,8.517393
